In [3]:
import json
import random
import subprocess
from datetime import datetime, timedelta, timezone

# ==== CONFIG ====
NUM_PATIENTS = 1
NUM_DAYS = 1


# ==== MODEL CALLER ====
def call_ollama(model, prompt):
    print(f"\n[DEBUG] Calling model: {model}")
    print(f"[DEBUG] Prompt:\n{prompt}")
    result = subprocess.run(
        ["ollama", "run", model],
        input=prompt.encode("utf-8"),
        stdout=subprocess.PIPE
    )
    output = result.stdout.decode("utf-8").strip()
    print(f"[DEBUG] Raw Model Output:\n{output}")
    return output

# ==== GENERATION FUNCTIONS ====
def generate_vitals():
    vitals = {
        "heart_rate": {"value": random.randint(60, 100), "unit": "bpm"},
        "spo2": {"value": random.randint(92, 99), "unit": "%"},
        "temperature": {"value": round(random.uniform(36.0, 38.0), 1), "unit": "°C"},
        "bp": {
            "systolic": random.randint(110, 160),
            "diastolic": random.randint(70, 100),
            "unit": "mmHg"
        }
    }
    print(f"[DEBUG] Generated Vitals: {vitals}")
    return vitals

def generate_symptoms(vitals):
    prompt = f"Patient vitals: {json.dumps(vitals)}"
    try:
        symptoms = json.loads(call_ollama("symptom_generator", prompt))
        if not symptoms:
            raise ValueError("Empty symptoms list")
        return symptoms
    except:
        print("[WARNING] symptom_generator failed, using fallback symptoms.")
        return ["fatigue"]

def generate_medications(symptoms):
    prompt = f"Patient symptoms: {json.dumps(symptoms)}"
    try:
        meds = json.loads(call_ollama("medication_planner", prompt))
        if not meds:
            raise ValueError("Empty medications list")
        return meds
    except:
        print("[WARNING] medication_planner failed, using fallback meds.")
        return [{"name": "Paracetamol", "dose": "500mg", "scheduled_time": "08:00", "compliance": "Taken"}]

def generate_nursing_note(symptoms, vitals, medications):
    prompt = f"""
    Symptoms: {symptoms}
    Vitals: {vitals}
    Medications: {medications}
    """
    return call_ollama("nursing_note_generator", prompt)

def generate_clinical_summary(nursing_note):
    prompt = f"Summarise this nursing note into 3–5 lines:\n{nursing_note}"
    return call_ollama("guardian_nurse", prompt)

# ==== MAIN RECORD GENERATOR ====
def generate_patient_record(patient_id, date):
    vitals = generate_vitals()
    symptoms = generate_symptoms(vitals)
    medications = generate_medications(symptoms)
    nursing_note = generate_nursing_note(symptoms, vitals, medications)
    clinical_summary = generate_clinical_summary(nursing_note)

    record = {
        "patient_id": f"P{patient_id:05d}",
        "age": random.randint(65, 95),
        "gender": random.choice(["Male", "Female"]),
        "timestamp": date.isoformat(),
        "nursing_note": nursing_note,
        "medications": medications,
        "vitals": vitals,
        "adls": {
            "steps_taken": random.randint(1000, 5000),
            "calorie_intake": random.randint(800, 2000),
            "sleep_hours": random.randint(4, 9)
        },
        "behaviour_tags": [],
        "emotion_analysis": {"tags": [], "confidence_scores": {}},
        "clinical_summary": clinical_summary,
        "entities_extracted": {},
        "baseline_stats": {
            "avg_heart_rate": None,
            "avg_spo2": None,
            "avg_sleep_hours": None,
            "avg_calorie_intake": None,
            "usual_diet_compliance": None
        },
        "alerts": []
    }
    print(f"\n[DEBUG] Final Patient Record:\n{json.dumps(record, indent=2)}\n")
    return record

# ==== DATASET GENERATION ====
def main():
    start_date = datetime.now(timezone.utc) - timedelta(days=NUM_DAYS)
    for pid in range(1, NUM_PATIENTS + 1):
        for d in range(NUM_DAYS):
            date = start_date + timedelta(days=d)
            generate_patient_record(pid, date)

if __name__ == "__main__":
    main()

[DEBUG] Generated Vitals: {'heart_rate': {'value': 100, 'unit': 'bpm'}, 'spo2': {'value': 96, 'unit': '%'}, 'temperature': {'value': 38.0, 'unit': '°C'}, 'bp': {'systolic': 158, 'diastolic': 82, 'unit': 'mmHg'}}

[DEBUG] Calling model: symptom_generator
[DEBUG] Prompt:
Patient vitals: {"heart_rate": {"value": 100, "unit": "bpm"}, "spo2": {"value": 96, "unit": "%"}, "temperature": {"value": 38.0, "unit": "\u00b0C"}, "bp": {"systolic": 158, "diastolic": 82, "unit": "mmHg"}}


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
Error: pull model manifest: file does not exist


[DEBUG] Raw Model Output:

[WARNING] symptom_generator failed, using fallback symptoms.

[DEBUG] Calling model: medication_planner
[DEBUG] Prompt:
Patient symptoms: ["fatigue"]


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
Error: pull model manifest: file does not exist
pulling manifest ⠙ 

[DEBUG] Raw Model Output:

[WARNING] medication_planner failed, using fallback meds.

[DEBUG] Calling model: nursing_note_generator
[DEBUG] Prompt:

    Symptoms: ['fatigue']
    Vitals: {'heart_rate': {'value': 100, 'unit': 'bpm'}, 'spo2': {'value': 96, 'unit': '%'}, 'temperature': {'value': 38.0, 'unit': '°C'}, 'bp': {'systolic': 158, 'diastolic': 82, 'unit': 'mmHg'}}
    Medications: [{'name': 'Paracetamol', 'dose': '500mg', 'scheduled_time': '08:00', 'compliance': 'Taken'}]
    


pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠼ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠧ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
Error: pull model manifest: file does not exist
pulling manifest ⠙ 

[DEBUG] Raw Model Output:


[DEBUG] Calling model: guardian_nurse
[DEBUG] Prompt:
Summarise this nursing note into 3–5 lines:



pulling manifest ⠹ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ 

[DEBUG] Raw Model Output:


[DEBUG] Final Patient Record:
{
  "patient_id": "P00001",
  "age": 84,
  "gender": "Male",
  "timestamp": "2026-03-21T11:21:23.514931+00:00",
  "nursing_note": "",
  "medications": [
    {
      "name": "Paracetamol",
      "dose": "500mg",
      "scheduled_time": "08:00",
      "compliance": "Taken"
    }
  ],
  "vitals": {
    "heart_rate": {
      "value": 100,
      "unit": "bpm"
    },
    "spo2": {
      "value": 96,
      "unit": "%"
    },
    "temperature": {
      "value": 38.0,
      "unit": "\u00b0C"
    },
    "bp": {
      "systolic": 158,
      "diastolic": 82,
      "unit": "mmHg"
    }
  },
  "adls": {
    "steps_taken": 1369,
    "calorie_intake": 1023,
    "sleep_hours": 6
  },
  "behaviour_tags": [],
  "emotion_analysis": {
    "tags": [],
    "confidence_scores": {}
  },
  "clinical_summary": "",
  "entities_extracted": {},
  "baseline_stats": {
    "avg_heart_rate": null,
    "avg_spo2": null,
    "avg_sleep_hours": null,
    "avg_calor

pulling manifest ⠧ pulling manifest 
Error: pull model manifest: file does not exist


None


In [34]:
#!pip install faster-whisper

In [24]:
# Install required package if you don't have it
#!pip install sounddevice scipyMHEALTH - UCI Machine Learning RepositoryMHEALTH - UCI Machine Learning RepositoryMHEALTH - UCI Machine Learning RepositoryMHEALTH - UCI Machine Learning RepositoryMHEALTH - UCI Machine Learning RepositoryMHEALTH - UCI Machine Learning RepositoryMHEALTH - UCI

In [5]:
import sounddevice as sd
import numpy as np
from scipy.io.wavfile import write

def record():
    # ----------------------------
    # Settings
    # ----------------------------
    duration = 10  # seconds to record
    fs = 16000    # sampling frequency (Hz)
    filename = "recorded_audio.wav"  # output file

    # ----------------------------
    # Record audio
    # ----------------------------
    print("Recording...")
    recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()  # Wait until recording is finished
    print("Recording finished!")

    # ----------------------------
    # Save as WAV file
    # ----------------------------
    # Convert to int16 for WAV format
    write(filename, fs, np.int16(recording * 32767))
    print(f"Audio saved as {filename}")
    speech_to_text()
    


In [13]:
#!pip install vosk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 4.7 MB/s eta 0:00:005.1 MB/s eta 0:00:01
  Created wheel for srt: filename=srt-3.5.3-py3-none-any.whl size=22428 sha256=6d0ba5b14173a2a537dcab2bbec7677f3a5250cbe354747ab2571e3db50968cc
  Stored in directory: /Users/tech_nauch/Library/Caches/pip/wheels/7e/75/5b/e1d5c3756631e4bda806f6cc9640153b39484bb6f7b0b8def3
Successfully built srt


In [7]:
import wave
import json
from vosk import Model, KaldiRecognizer

def speech_to_text():
    

    # Load the Vosk model
    model_path = r"C:\Users\User\Downloads\vosk-model-small-en-us-0.15"
    model = Model(model_path)

    # Open recorded audio
    wf = wave.open("recorded_audio.wav", "rb")

    # Initialize recognizer
    rec = KaldiRecognizer(model, wf.getframerate())

    transcription = ""

    # Transcribe
    while True:
        data = wf.readframes(4000)

        if len(data) == 0:
            break

        if rec.AcceptWaveform(data):
            result = json.loads(rec.Result())
            transcription += result.get("text", "") + " "

    # Final result
    result = json.loads(rec.FinalResult())
    transcription += result.get("text", "")
    
    text_box.insert(tk.END, transcription)

    return transcription


# Run transcription


In [ ]:
import tkinter as tk
from tkinter import messagebox



root = tk.Tk()
root.title("Recorder")
root.geometry("300x400")

# Bring window to front
root.attributes("-topmost", True)

# Text box
text_box = tk.Text(root, height=10, width=50)
text_box.pack(fill="both",expand=True,padx=5, pady=(5,0))

# Create button
record_button = tk.Button(root, text="Record", command=record, height=2, width=15)
record_button.pack(side="bottom", pady=40)

root.mainloop()

In [1]:
from google import genai

client = genai.Client(api_key="YOUR_API_KEY")

def correct_transcript(transcript: str) -> str:
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=f"""
Correct the grammar and punctuation of this transcript.
Add full stops and commas.
Keep the meaning same.

Transcript:
{transcript}

Return only corrected text.
"""
        )
        return response.text.strip()

    except Exception as e:
        print("API Error:", e)
        return transcript

In [ ]:
transcript = record()
corrected_text = correct_transcript(transcript)

print("Original:")
print(transcript)

print("\nCorrected:")
print(corrected_text)